In [1]:
import os
import polars as pl
from src.utils.config import setup_clickhouse_client
from src.utils.helper import convert_row
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

In [2]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pl.DataFrame(flights_result, schema=flights_columns, orient='row')

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

Total flight records retrieved: 129442


id,insert_time,flight_type,status,iata_number,airline_name,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,codeshared_airline,codeshared_flight_number
str,str,str,str,str,str,str,str,i64,str,str,i64,str,str
"""6d6eb7ea-694b-4050-b2bf-2a5cb5…","""2026-04-10T14:49:47.510000+08:…","""arrival""","""landed""","""ua1""","""united airlines""","""2026-02-27T14:35:00""","""2026-02-27T14:57:00""",22,"""2026-03-01T00:01:00""","""2026-02-28T23:35:00""",0,"""""",""""""
"""3c71e857-3e4f-41cb-9fcf-0130d2…","""2026-04-10T14:50:16.484000+08:…","""arrival""","""landed""","""sq27""","""singapore airlines""","""2026-02-28T00:45:00""","""2026-02-28T01:18:00""",33,"""2026-03-01T09:45:00""","""2026-03-01T09:33:00""",0,"""""",""""""
"""2d6be94c-3555-4311-95aa-13c80e…","""2026-04-10T14:50:16.484000+08:…","""arrival""","""landed""","""sq21""","""singapore airlines""","""2026-02-28T01:35:00""","""2026-02-28T02:03:00""",29,"""2026-03-01T09:45:00""","""2026-03-01T09:25:00""",0,"""""",""""""
"""70b0b485-efc0-4753-8280-a4c9e5…","""2026-04-10T14:50:24.761000+08:…","""arrival""","""landed""","""sq31""","""singapore airlines""","""2026-02-28T01:40:00""","""2026-02-28T02:10:00""",31,"""2026-03-01T11:05:00""","""2026-03-01T10:44:00""",0,"""""",""""""
"""d70fe84e-694d-4020-97e6-6a879e…","""2026-04-10T14:50:28.880000+08:…","""arrival""","""landed""","""ua29""","""united airlines""","""2026-02-28T02:35:00""","""2026-02-28T04:13:00""",98,"""2026-03-01T12:15:00""","""2026-03-01T12:49:00""",34,"""""",""""""


In [3]:
weather_query = "SELECT * FROM historical_weather_data order by date_observed"

weather_result = client.query(weather_query).result_rows
weather_result = [convert_row(row) for row in weather_result] 
weather_columns = client.query(weather_query).column_names

weather_pdf = pl.DataFrame(weather_result, schema=weather_columns, orient='row')

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

Total weather records retrieved: 977


id,insert_time,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,min_temp,max_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
str,str,str,f64,f64,i64,i64,f64,f64,f64,i64,i64,f64,f64,str,str
"""5cc74c3f-b81e-48c6-ac3f-0c2d65…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T16:00:00""",27.79,31.72,1009,81,26.92,27.97,3.09,30,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""ebec1e01-2b88-42cb-8772-f39638…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T17:00:00""",27.22,30.79,1009,85,26.92,27.74,2.57,70,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""eb52ac12-8740-498b-ae0d-aab525…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T18:00:00""",27.22,30.9,1008,86,26.92,27.74,2.57,10,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""7a20d0bb-1920-4d87-91cc-0afaff…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T19:00:00""",27.22,30.9,1007,86,25.92,27.74,2.57,350,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""b2b7e1f2-5e3d-445f-8c5c-a3363c…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T20:00:00""",26.93,30.34,1007,88,25.92,27.18,2.57,340,75,0.0,0.0,"""Clouds""","""broken clouds"""


In [4]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.filter(~pl.col('status').is_in(valid_statuses)) \
                    .unique(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                    'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time'],
                                    keep='first')

print(f"Flight records after filtering: {len(flights_filtered)}")

Prepared flights and weather data for merging.
Flight records after filtering: 126064


In [5]:
# Create hour columns for both departure and arrival times
flights_filtered = flights_filtered.with_columns(
    pl.col('dep_scheduled_time').str.to_datetime().dt.truncate('1h').alias('dep_hour'),
    pl.col('arr_scheduled_time').str.to_datetime().dt.truncate('1h').alias('arr_hour')
)

# Split into departure and arrival flights for conditional merging 
departure_flights = flights_filtered.filter(pl.col('flight_type') == 'departure')
arrival_flights = flights_filtered.filter(pl.col('flight_type') == 'arrival')

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")
departure_flights.head(3)
arrival_flights.head(3)


Departure flights: 63390, Arrival flights: 62674


id,insert_time,flight_type,status,iata_number,airline_name,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,codeshared_airline,codeshared_flight_number,dep_hour,arr_hour
str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,datetime[μs],datetime[μs]
"""9ec476d4-93a5-4397-a745-b5dabe…","""2026-04-10T14:57:51.344000+08:…","""arrival""","""landed""","""ua7701""","""united airlines""","""2026-03-07T05:00:00""","""2026-03-07T04:59:00""",0,"""2026-03-07T07:50:00""","""2026-03-07T07:10:00""",0,"""singapore airlines""","""sq939""",2026-03-07 05:00:00,2026-03-07 07:00:00
"""efb0b806-394d-4827-8ec6-8e7a66…","""2026-04-10T14:56:29.192000+08:…","""arrival""","""landed""","""tr487""","""scoot""","""2026-03-06T09:30:00""","""2026-03-06T09:42:00""",13,"""2026-03-06T10:50:00""","""2026-03-06T10:57:00""",8,"""""","""""",2026-03-06 09:00:00,2026-03-06 10:00:00
"""351d8933-c0a5-43f3-832a-f6d2ef…","""2026-04-10T15:56:02.277000+08:…","""arrival""","""landed""","""sq8551""","""singapore airlines""","""2026-03-17T11:10:00""","""2026-03-17T11:26:00""",17,"""2026-03-17T12:25:00""","""2026-03-17T12:21:00""",0,"""scoot""","""tr469""",2026-03-17 11:00:00,2026-03-17 12:00:00


In [6]:
weather_pdf = weather_pdf.with_columns(
    pl.col('date_observed').str.to_datetime().dt.truncate('1h').alias('weather_hour') 
)

print(f"Weather records by hour: {len(weather_pdf)}")

weather_pdf.head(3)

Weather records by hour: 977


id,insert_time,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,min_temp,max_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc,weather_hour
str,str,str,f64,f64,i64,i64,f64,f64,f64,i64,i64,f64,f64,str,str,datetime[μs]
"""5cc74c3f-b81e-48c6-ac3f-0c2d65…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T16:00:00""",27.79,31.72,1009,81,26.92,27.97,3.09,30,75,0.0,0.0,"""Clouds""","""broken clouds""",2026-02-28 16:00:00
"""ebec1e01-2b88-42cb-8772-f39638…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T17:00:00""",27.22,30.79,1009,85,26.92,27.74,2.57,70,75,0.0,0.0,"""Clouds""","""broken clouds""",2026-02-28 17:00:00
"""eb52ac12-8740-498b-ae0d-aab525…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T18:00:00""",27.22,30.9,1008,86,26.92,27.74,2.57,10,75,0.0,0.0,"""Clouds""","""broken clouds""",2026-02-28 18:00:00


In [7]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour
if len(departure_flights) > 0:
    merged_departures = departure_flights.join(
        weather_pdf,
        how='left',
        left_on='dep_hour',
        right_on='weather_hour',
        maintain_order='left'
    )

    merged_departures= merged_departures.drop(['arr_scheduled_time', 'arr_actual_time', 'arr_delay_mins']) \
        .sort(by=['dep_scheduled_time', 'dep_actual_time'], nulls_last=True) \
        .rename({'dep_scheduled_time': 'scheduled_time', 'dep_actual_time': 'actual_time', 'dep_delay_mins': 'delay_mins'})

    print(f"Merged departure flights: {len(merged_departures)}")
    merged_departures.head(3)

Merged departure flights: 64313


In [8]:
# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = arrival_flights.join(
        weather_pdf,
        how='left',
        left_on='arr_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    
    merged_arrivals= merged_arrivals.drop(['dep_scheduled_time', 'dep_actual_time', 'dep_delay_mins']) \
    .sort(by=['arr_scheduled_time', 'arr_actual_time'], nulls_last=True) \
    .rename({'arr_scheduled_time': 'scheduled_time', 'arr_actual_time': 'actual_time', 'arr_delay_mins': 'delay_mins'})

    print(f"Merged arrival flights: {len(merged_arrivals)}")
    merged_arrivals.head(3)

Merged arrival flights: 63897


In [9]:
# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pl.concat([merged_departures, merged_arrivals], how='vertical') \
            .sort(by=['scheduled_time', 'actual_time'], nulls_last=True)
    # print(combined_flights.columns)
    combined_flights = combined_flights.drop(['id', 'insert_time', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number'])
else:
    combined_flights = pl.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)


 Combined flights with weather data: 128210 rows


flight_type,status,scheduled_time,actual_time,delay_mins,dep_hour,arr_hour,id_right,insert_time_right,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,min_temp,max_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
str,str,str,str,i64,datetime[μs],datetime[μs],str,str,str,f64,f64,i64,i64,f64,f64,f64,i64,i64,f64,f64,str,str
"""arrival""","""landed""","""2026-03-01T00:01:00""","""2026-02-28T23:14:00""",0,2026-02-28 18:00:00,2026-03-01 00:00:00,"""c3062f11-c2ea-49d7-aab4-529e04…","""2026-04-10T12:45:38.549000+08:…","""2026-03-01T00:00:00""",26.36,26.36,1008,90,25.92,27.18,2.57,40,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""arrival""","""landed""","""2026-03-01T00:01:00""","""2026-02-28T23:14:00""",0,2026-02-28 18:00:00,2026-03-01 00:00:00,"""c3062f11-c2ea-49d7-aab4-529e04…","""2026-04-10T12:45:38.549000+08:…","""2026-03-01T00:00:00""",26.36,26.36,1008,90,25.92,27.18,2.57,40,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""arrival""","""landed""","""2026-03-01T00:01:00""","""2026-02-28T23:35:00""",0,2026-02-27 14:00:00,2026-03-01 00:00:00,"""c3062f11-c2ea-49d7-aab4-529e04…","""2026-04-10T12:45:38.549000+08:…","""2026-03-01T00:00:00""",26.36,26.36,1008,90,25.92,27.18,2.57,40,75,0.0,0.0,"""Clouds""","""broken clouds"""


In [21]:
# Machine Learning and prediction below
#
#

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# Select features and target
feature_list = ["current_temp", "feels_like_temp", "pressure_hPa", "humidity_pct", "wind_speed_ms", "wind_deg", "cloudiness_pct", "rain_1h", "weather_main"]
features_var = combined_flights.select(feature_list)
independent_var = combined_flights["delay_mins"]

In [ ]:
y_statement = independent_var.to_numpy()
y_binary = (y_statement > 15).astype(int)
class_counts = np.bincount(y_binary)

print(f"Delays > 15 min: {class_counts[0]}, otherwise: {class_counts[1]}")

print(features_var.dtypes)
features_var = features_var.to_dummies(columns=["weather_main"])
X = combined_flights.select(features_var).to_numpy()
X = np.nan_to_num(X, nan=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

sample_weights = compute_sample_weight('balanced', y_train)

Delays > 15 min: 75010, otherwise: 53200
[Float64, Float64, Int64, Int64, Float64, Int64, Int64, Float64, String]
Sample weights: [1.20679593 1.20679593 0.85370888 ... 0.85370888 1.20679593 0.85370888]


In [25]:
#  Test binary class problem with random forest decision trees
#
#

rf_results = []

model = RandomForestClassifier(n_estimators=250, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
bal_acc_score=balanced_accuracy_score(y_test, y_pred)
class_rpt = classification_report(y_test, y_pred, target_names=['delayed', 'not delayed'])

rf_results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'bal_acc_score': bal_acc_score,
    'class_rpt': class_rpt
})

results_rf = pd.DataFrame(rf_results)
print(results_rf[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].to_string())

print("\n\nSummary of Random Forest Algorithm for delay prediction:")
best = results_rf.iloc[0]
print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f'Balanced accuracy score: {balanced_accuracy_score(y_test, y_pred):.2%}')
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")
print(f"Classification report: {class_rpt}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

   cv_mean_accuracy    cv_std  test_accuracy  precision    recall  f1_score
0           0.64174  0.003249       0.616255   0.621935  0.616255   0.61817


Summary of Random Forest Algorithm for delay prediction:
Cross-Val Accuracy (mean±std): 64.17% ± 0.32%
Test Accuracy: 61.63%
Balanced accuracy score: 61.21%
Precision: 62.19%
Recall: 61.63%
f1_score: 61.82%
Classification report:               precision    recall  f1-score   support

     delayed       0.68      0.64      0.66     14938
 not delayed       0.54      0.59      0.56     10704

    accuracy                           0.62     25642
   macro avg       0.61      0.61      0.61     25642
weighted avg       0.62      0.62      0.62     25642


Confusion Matrix:
 [[9519 5419]
 [4421 6283]]


In [28]:
#  Test binary class problem with gradient boost
#
#

gb_results = []

# Model
model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_depth=10,
    min_samples_leaf= 10,
    random_state=42,
    warm_start=True
)

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
model.fit(X_train, y_train, sample_weights)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
bal_acc_score=balanced_accuracy_score(y_test, y_pred)
class_rpt = classification_report(y_test, y_pred, target_names=['delayed', 'not delayed'])

# Metrics
gb_results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'bal_acc_score': bal_acc_score,
    'class_rpt': class_rpt
})

results_gb = pd.DataFrame(gb_results)
print(results_rf[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].to_string())

print('\n\n Summary of Random Forest Algorithm for delay prediction:')
best = results_gb.iloc[0]

print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f'Balanced accuracy score: {balanced_accuracy_score(y_test, y_pred):.2%}')
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")
print(f"Classification report: {class_rpt}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

   cv_mean_accuracy    cv_std  test_accuracy  precision    recall  f1_score
0           0.64174  0.003249       0.616255   0.621935  0.616255   0.61817


 Summary of Random Forest Algorithm for delay prediction:
Cross-Val Accuracy (mean±std): 64.08% ± 0.28%
Test Accuracy: 61.79%
Balanced accuracy score: 61.05%
Precision: 62.06%
Recall: 61.79%
f1_score: 61.90%
Classification report:               precision    recall  f1-score   support

     delayed       0.68      0.66      0.67     14938
 not delayed       0.54      0.57      0.55     10704

    accuracy                           0.62     25642
   macro avg       0.61      0.61      0.61     25642
weighted avg       0.62      0.62      0.62     25642


Confusion Matrix:
 [[9790 5148]
 [4649 6055]]
